**Multimodel Fine-Tuning**

https://huggingface.co/datasets/liuhaotian/LLaVA-Instruct-150K

https://huggingface.co/datasets/HuggingFaceM4/ChartQA

https://huggingface.co/datasets/nlphuji/flickr30k

https://huggingface.co/datasets/unsloth/LaTeX_OCR

https://huggingface.co/datasets/sunny199/iphone5_vlm_img

https://huggingface.co/datasets/sunny199/iphone5_vlm


In [ ]:
!pip -q install "transformers==4.57.1" --upgrade
!pip -q install --no-deps trl==0.22.2
!pip -q install unsloth unsloth_zoo bitsandbytes accelerate peft triton
!pip install -U transformers datasets peft accelerate bitsandbytes trl
!pip -q install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install -U datasets huggingface_hub pillow

In [ ]:
import unsloth
import os
import torch
from dataclasses import dataclass
from typing import Dict
from datasets import load_dataset, Dataset, Features, Value, Image
from huggingface_hub import login
from google.colab import userdata
from transformers import TextStreamer
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)

MODEL_REGISTRY = {
    "qwen2_vl_2b": "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    "qwen25_vl_3b": "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",
    "llava15_7b": "unsloth/llava-1.5-7b-hf-bnb-4bit",
    "pixtral_12b": "unsloth/Pixtral-12B-2409-bnb-4bit",
    "medgemma_4b": "unsloth/MedGemma-4B-Vision-Instruct-bnb-4bit",
}

DATASET_REGISTRY = {
    "latex_ocr": {
        "name": "unsloth/LaTeX_OCR",
        "split": "train",
        "image_key": "image",
        "text_key": "text",
        "instruction": "Write the LaTeX representation for this image."
    },
    "flickr30k": {
        "name": "nlphuji/flickr30k",
        "split": "train",
        "image_key": "image",
        "text_key": "caption",
        "instruction": "Describe the image."
    },

    "iphone_custom": {
    "name": "Emon34/iphone5_vlm",
    "split": "train",
    "image_key": "image",
    "text_key": "text",
    "instruction": "Describe this iPhone product image in one sentence."
  },
}

In [ ]:
@dataclass
class VisionFTConfig:
  model_key: str = "qwen2_vl_2b"
  dataset_key: str = "latex_ocr"
  subset_rows: int = 150
  eval_ratio: float = 0.1
  seed: int = 3407

  r: int = 16
  lora_alpha: int = 16
  lora_dropout: float = 0.0

  per_device_train_batch_size: int = 2
  gradient_accumulation_steps: int = 4
  num_train_epochs: int = 2
  learning_rate: float = 2e-4
  logging_steps: int = 10
  weight_decay: float = 0.001
  max_length: int = 2048

  output_dir: str = "outputs"
  save_dir: str = "vlm_lora_output"

dataset = load_dataset("HuggingFaceM4/ChartQA")
train_ds = load_dataset("HuggingFaceM4/ChartQA", split="train")
dataset, train_ds[0]

In [ ]:
class VisionFineTuner:
  def __init__(self, cfg: VisionFTConfig):
    self.cfg = cfg
    self.model_name = MODEL_REGISTRY[cfg.model_key]
    self.dataset_info = DATASET_REGISTRY[cfg.dataset_key]

  def load_model(self):
    self.model, self.tokenizer = FastVisionModel.from_pretrained(
        self.model_name,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth"
    )

    self.model = FastVisionModel.get_peft_model(
        self.model,
        finetune_vision_layers = True,
        finetune_language_layers = True,
        finetune_attention_modules= True,
        finetune_mlp_modules = True,
        r=self.cfg.r,
        lora_alpha=self.cfg.lora_alpha,
        lora_dropout=self.cfg.lora_dropout,
        bias="none",
        random_state=self.cfg.seed,
    )

    return self

  def prepare_data(self):
    raw = load_dataset(
        self.dataset_info["name"],
        split=self.dataset_info["split"]
    )

    raw = raw.select(range(min(self.cfg.subset_rows, len(raw))))
    instruction = self.dataset_info["instruction"]
    image_key = self.dataset_info["image_key"]
    text_key = self.dataset_info["text_key"]

    def format_sample(example):
      return {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": instruction},
                            {"type": "image", "image": example[image_key]},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [
                            {"type": "text", "text": str(example[text_key])}
                        ],
                    },
                ]
            }

    ds = raw.map(format_sample, remove_columns=raw.column_names)
    splits = ds.train_test_split(
        test_size=self.cfg.eval_ratio,
        seed=self.cfg.seed,
    )

    self.train_ds = splits["train"]
    self.eval_ds = splits["test"]

    return self

  def build_trainer(self):
    FastVisionModel.for_training(self.model)
    args = SFTConfig(
        per_device_train_batch_size=self.cfg.per_device_train_batch_size,
        gradient_accumulation_steps=self.cfg.gradient_accumulation_steps,
        num_train_epochs=self.cfg.num_train_epochs,
        learning_rate=self.cfg.learning_rate,
        logging_steps=self.cfg.logging_steps,
        optim="adamw_8bit",
        weight_decay=self.cfg.weight_decay,
        seed=self.cfg.seed,
        output_dir=self.cfg.output_dir,
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=self.cfg.max_length,
    )

    self.trainer = SFTTrainer(
        model = self.model,
        tokenizer = self.tokenizer,
        data_collator = UnslothVisionDataCollator(self.model, self.tokenizer),
        train_dataset = self.train_ds,
        eval_dataset = self.eval_ds,
        args=args
    )

    return self

  def train(self):
    print("Training for", self.cfg.num_train_epochs, "epochs")
    self.trainer.train()

  def save(self):
    os.makedirs(self.cfg.save_dir, exist_ok=True)
    self.model.save_pretrained(self.cfg.save_dir)
    self.tokenizer.save_pretrained(self.cfg.save_dir)
    print("Saved to:", self.cfg.save_dir)

  def quick_infer(self, sample_index=0):
    FastVisionModel.for_inference(self.model)

    raw = load_dataset(
        self.dataset_info["name"],
        split=self.dataset_info["split"]
    )
    image = raw[sample_index][self.dataset_info["image_key"]]
    message = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": self.dataset_info["instruction"]}
        ]}
    ]

    input_text = self.tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True
    )

    inputs = self.tokenizer(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to("cuda")

    streamer = TextStreamer(self.tokenizer, skip_prompt=True)

    self.model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=128,
        temperature=1.2
    )

cfg = VisionFTConfig(
  model_key="qwen2_vl_2b",
  dataset_key="latex_ocr",
)

trainer = (
  VisionFineTuner(cfg)
  .load_model()
  .prepare_data()
  .build_trainer()
)

trainer.train()
trainer.save()
trainer.quick_infer()

In [ ]:
userdata.get('')
login(token="")

In [ ]:
IMG_DIR = "./iphone_imgs"
image_files = sorted([
    os.path.join(IMG_DIR, f)
    for f in os.listdir(IMG_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
])

captions = [
    "A white iPhone shown from the back and front on a white background.",
    "A dark blue iPhone shown from the back with a side view on a transparent background.",
    "An orange iPhone shown from the back and front on a white background.",
    "Three iPhones in white, orange, and dark blue shown together from the back.",
    "An orange iPhone shown from the back on a transparent background.",
]

rows = []
for img_path, cap in zip(image_files, captions):
    rows.append({
        "image": img_path,
        "text": cap,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instruction},
                    {"type": "image", "image": img_path},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": cap},
                ],
            },
        ],
    })


features = Features([
    "image": Image(),
    "text": Value("string"),
    "messages": Value("string"),
])

ds = Dataset.from_list(rows, features=features)
splits = ds.train_test_split(test_size=1, seed=3407)
REPO_ID = "Emon3469/new_vlm_img"
splits["train"].push_to_hub(REPO_ID, split="train")
splits["test"].push_to_hub(REPO_ID, split="test")

print("Uploaded:", REPO_ID)

In [ ]:
dataset = load_dataset("HuggingFaceM4/ChartQA")
train_ds = load_dataset("HuggingFaceM4/ChartQA", split="train")
dataset, train_ds[0]

## **Embedding and Embedding Fine-Tuning**

In [ ]:
!pip install sentence-transformers torch
!pip install -U langchain
!pip install -U langchain-openai
%pip install -qU langchain-community pypdf
!pip install -q faiss-cpu
!pip install mteb

In [ ]:
import re
import os
import zipfile
import mteb
import numpy as np
from typing import Dict, Any, Optional
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from datasets import load_from_disk
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores.faiss import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

warnings.filterwarnings("ignore", category=DeprecationWarning)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
text = "Embedding fine-tuning improves retrieval quality in RAG systems."
embedding = model.encode(text)
print("Embedding Shape:", embedding.shape)
print("First 10 Values:", np.round(embedding[:10], 6))
print(embedding)

In [ ]:
def consine_sim(a, b):
  return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

documents = [
    "Embedding fine-tuning improves retrieval in RAG.",
    "LoRA reduces GPU memory usage.",
    "My car needs fuel.",
    "Vector databases store embeddings.",
]

query = "How to improve retrieval in RAG?"
doc_embds = model.encode(documents, convert_to_numpy=True)
query_embds = model.encode(query, convert_to_numpy=True)
scores = [consine_sim(query_embds, embd) for embd in doc_embds]
print(query)
for doc, score in zip(documents, scores):
    print(f"Score: {score:.4f}  |  {doc}")

In [ ]:
pairs = [
    ("Embedding improves semantic search",
     "Fine-tuned embeddings enhance retrieval"),

    ("Embedding improves semantic search",
     "I bought a new car yesterday"),
]

for s1, s2 in pairs:
  embd1, embd2 = model.encode([s1, s2], normalize_embeddings=True)
  score = consine_sim(embd1, embd2)
  print(f"\nSentence 1: {s1}")
  print(f"Sentence 2: {s2}")
  print("Cosine Similarity:", round(score, 4))

In [ ]:
documents = [
    "Embedding fine-tuning improves retrieval in RAG.",
    "LoRA reduces GPU memory usage.",
    "My car needs fuel.",
    "Vector databases store embeddings.",
]

def tokenize(text):
  return re.findall(r"[a-z0-9]+", text.lower())

query = "How to improve retrieval in RAG?"
query_tokens = tokenize(query)
query_tokens

In [ ]:
print("Query tokens:", query_tokens)
print("\n=== Keyword Search Scores ===")

for doc in documents:
  doc_tokens = tokenize(doc)
  matches = set(query_tokens) & set(doc_tokens)
  score = len(matches)

  print(f"Score: {score} | Matched: {matches} | {doc}")

In [ ]:
documents = [
    "Embedding fine-tuning improves retrieval in RAG.",
    "LoRA reduces GPU memory usage.",
    "My car needs fuel.",
    "Vector databases store embeddings.",
]

query = "How to improve retrieval in RAG?"
def tokenize(text: str):
    return re.findall(r"[a-z0-9]+", text.lower())
tokenized_corpus = [tokenize(doc) for doc in documents]
tokenized_query  = tokenize(query)

tokenized_corpus, tokenized_query

In [ ]:
bm25 = BM25Okapi(tokenized_corpus)
scores = bm25.get_scores(tokenized_query)
print("=== BM25 Scores (all docs) ===")
for doc, score in zip(documents, scores):
    print(f"{score:.4f} | {doc}")

In [ ]:
def eval_stsbenchmark(model_name: str, batch_size: int = 16, normalize_embeddings: bool = True, languages: Optional[list[str]] = None) -> Dict[str, any]:
  if languages is None:
    languages = ["eng"]
  model = mteb.get_model(model_name)
  tasks = mteb.get_tasks(tasks=["STSBenchmark"], languages=languages)
  results = mteb.evaluate(
      model,
      tasks,
      encode_kwargs= {
          "batch_size": batch_size,
          "normalize_embeddings": normalize_embeddings,
      },
  )

  tr = next((t for t in results.task_results if t.task_name == "STSBenchmark"), None)
  if tr is None:
    raise RuntimeError("STSBenchmark TaskResult not found in results.")

  test_entry = tr.scores["test"][0]
  out = {
    "model_name": results.model_name,
    "model_revision": results.model_revision,
    "main_score": float(test_entry["main_score"]),
    "spearman": float(test_entry["spearman"]),
    "pearson": float(test_entry["pearson"]),
    "cosine_spearman": float(test_entry["cosine_spearman"]),
    "cosine_pearson": float(test_entry["cosine_pearson"]),
    "euclidean_spearman": float(test_entry["euclidean_spearman"]),
    "manhattan_spearman": float(test_entry["manhattan_spearman"]),
    "languages": test_entry.get("languages"),
    "hf_subset": test_entry.get("hf_subset"),
    "raw": results,
  }

  print("\n=== STSBenchmark (MTEB) ===")
  print(f"Model: {out['model_name']}")
  print(f"Revision: {out['model_revision']}")
  print(f"Main (Spearman): {out['main_score']:.4f}")
  print(f"Spearman: {out['spearman']:.4f} | Pearson: {out['pearson']:.4f}")
  print(
    "Cosine Spearman: {cs:.4f} | Euclidean Spearman: {es:.4f} | Manhattan Spearman: {ms:.4f}".format(
        cs=out["cosine_spearman"],
        es=out["euclidean_spearman"],
        ms=out["manhattan_spearman"],
      )
  )

  return out

r1 = eval_stsbenchmark("sentence-transformers/all-MiniLM-L6-v2")
r2 = eval_stsbenchmark("BAAI/bge-base-en-v1.5")
r3 = eval_stsbenchmark("intfloat/e5-base-v2")

results = [r1, r2, r3]
for r in sorted(results, key=lambda x: x["main_score"], reverse=True):
  print(f"{r['model_name']}: {r['main_score']:.4f}")

In [ ]:
zip_path = "/content/pharma_ft_data.zip"
extract_to = "unzipped_data"

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
train_dataset = load_from_disk("/content/unzipped_data/pharma_ft_data/train_pairs")

print("Dataset loaded:", train_dataset)
print("Columns:", train_dataset.column_names)

In [ ]:
loss = MultipleNegativesRankingLoss(model=model)

args = SentenceTransformerTrainingArguments(
    output_dir="models/pharma-embedding-ft",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=10,
    save_strategy="epoch",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

trainer.train()

In [ ]:
model.save_pretrained("models/pharma-embedding-ft")
def zip_folder_manual(folder_path: str, output_zip_path: str):
    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname)

    print(f"[OK] Folder zipped at: {output_zip_path}")


zip_folder_manual("/content/models", "model_backup.zip")

In [ ]:
embeddings_model = SentenceTransformer("/content/models/pharma-embedding-ft")
sentences = [
    "How to improve RAG retrieval?",
    "LoRA reduces GPU memory."
]
embeddings = embeddings_model.encode(sentences, normalize_embeddings=True)
vectorstore = FAISS.from_embeddings(documents, embeddings)
query = "What are the contraindications?"
results = vectorstore.similarity_search(query, k=3)

for i, r in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(r.page_content[:500])

In [ ]:
loader = PyPDFLoader("/content/pharma_demo.pdf")
documents = loader.load()
len(documents)

In [ ]:
MODEL_PATH = "/content/models/pharma-embedding-ft"

embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_PATH,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.from_documents(documents, embeddings)
vectorstore.save_local("faiss_index_")
retriever = vectorstore.as_retriever()
results = vectorstore.similarity_search("What are the warnings and precautions?", k=1)

for r in results:
  print("----")
  print(r.page_content)

In [ ]:
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
llm = ChatOpenAI(model="gpt-4o-mini")
llm.invoke("hi")

In [ ]:
prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant.
       Use the following context to answer the question. If you don't know, say you don't know.

       Context:
       {context}

       Question:
       {question}

       Answer:"""
)

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What are the contraindications for Amoxycillin 500 mg?")

### **LORA & QLORA Fine-tuning**

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./TinyLlama"

In [ ]:
dataset = load_dataset("Abirate/english_quotes")
dataset

In [ ]:
def format_example(example):
  quote = example["quote"].strip()
  tags = ", ".join(example["tags"]) if isinstance(example["tags"], list) else str(example["tags"])

  text = f"""<bos><start_of_turn>user
            Generate relevant tags for the following quote.

            Quote:
            {quote}<end_of_turn>
            <start_of_turn>model
            {tags}<end_of_turn>"""

  return {"text": text}

train_dataset = dataset["train"].map(format_example)
train_dataset, train_dataset['text'][0], train_dataset['text'][1]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
total_params = model.num_parameters()
trainable_params = model.num_parameters(only_trainable=True)
config = model.peft_config["default"]

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Trainable %:", 100 * trainable_params / total_params)
print(model.peft_config)
print("LoRA rank:", config.r)
print("LoRA alpha:", config.lora_alpha)
print("LoRA dropout:", config.lora_dropout)
print("Target modules:", config.target_modules)
print("Task type:", config.task_type)
print(base_model.config.quantization_config)
print("Loaded in 4-bit:", base_model.is_loaded_in_4bit)
print("Device map:", getattr(base_model, "hf_device_map", None))
print("Memory footprint in GB:", model.get_memory_footprint() / 1024**3)

for name, param in model.named_parameters():
  if param.requires_grad:
    print(name, param.shape, param.dtype)

for name, param in model.named_parameters():
  print(name, param.dtype, param.device, param.requires_grad)
  break

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    report_to="none",
    remove_unused_columns=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)

trainer.train()

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/adapter")

In [ ]:
from peft import PeftModel
base_model = MODEL_NAME
inference_model = PeftModel.from_pretrained(base_model, f"{OUTPUT_DIR}/adapter")
inference_model.eval()

In [ ]:
prompt = """<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
If you want to achieve greatness, stop asking for permission.<end_of_turn>
<start_of_turn>model
"""

inputs = tokenizer(prompt, return_tensors="pt").to(inference_model.device)

with torch.no_grad():
  output = inference_model.generate(
      **inputs,
      max_new_tokens=256,
      do_sample=True,
      temperature=0.7,
      top_k=50,
      top_p=0.9,
  )

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
merged_model = inference_model.merge_and_unload()
tokenizer.save_pretrained("./merged_full_model")
merged_model.save_pretrained("./merged_full_model")